In [1]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("smoke-test")

with mlflow.start_run(run_name="hello"):
    mlflow.log_param("model", "fake")
    mlflow.log_metric("spearman", 0.42)

print("logged")

c:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/08/08 09:34:27 INFO mlflow.tracking.fluent: Experiment with name 'smoke-test' does not exist. Creating a new experiment.


🏃 View run hello at: http://127.0.0.1:5000/#/experiments/2/runs/914cb10ad9cd4ed9b34cb436a0a2d0fd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
logged


In [1]:
import pandas as pd
df = pd.read_parquet(r"C:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot\data\history\all_seasons_fixed.parquet")
d = df[df["season"] == "2025-26"]

print("GWs present:", sorted(d["round"].unique()))
print("Rows missing value:", d["value"].isna().sum())

# does price actually move? if it never changes, sell-price logic is moot
piv = d.groupby("element")["value"].agg(["min", "max", "nunique"])
print("\nPlayers whose price never moved:", (piv["nunique"] == 1).sum())
print("Players whose price moved:      ", (piv["nunique"] > 1).sum())
print("Biggest rise (tenths):", (piv["max"] - piv["min"]).max())

GWs present: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38)]
Rows missing value: 0

Players whose price never moved: 241
Players whose price moved:       600
Biggest rise (tenths): 15


In [2]:
import pandas as pd
p = pd.read_parquet(r"C:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot\data\predictions_2526.parquet")
g1 = p[p["gw"] == 1]
print(g1[g1["name"].str.contains("Haaland|Salah", case=False)][["name","team","e_points","e_minutes"]])

                 name       team  e_points  e_minutes
14200   Mohamed Salah  Liverpool  7.963638  71.026191
16007  Erling Haaland   Man City  6.599158  73.052870


In [3]:
import pandas as pd
df = pd.read_parquet(r"C:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot\data\history\all_seasons_fixed.parquet")
d = df[df["season"] == "2025-26"]
print([c for c in d.columns if c in
  ["element","round","GW","minutes","total_points","position","team","value"]])
print(d[d["round"]==1][["element","minutes","total_points"]].head(3))

['element', 'minutes', 'round', 'total_points', 'value', 'GW', 'position', 'team']
        element  minutes  total_points
224143      541       90             6
224144       57        0             0
224145       87        0             0


In [1]:
import pandas as pd
w = pd.read_parquet(r"C:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot\data\walkforward_2526.parquet")
print(w.columns.tolist())
print(w.shape)
print(sorted(w["gw"].unique()))
print(w.head(3))

['element', 'gw', 'name', 'position', 'team', 'minutes', 'actual_points', 'player_id', 'understat_id', 'p_start', 'p60', 'e_minutes', 'understat_id_num', 'understat_id_r', 'npxg90', 'xa90', 'team_lambda', 'opp_lambda', 'p_cs', 'p_dc_hit', 'minutes_frac', 'fixture_scale', 'e_goals', 'e_assists', 'pts_goals', 'pts_assists', 'p_60plus', 'p_play_any', 'pts_appear', 'pts_cs', 'pts_dc', 'e_points_core', 'pred_bps', 'exp_bonus', 'e_points']
(29338, 35)
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38)]
   element  gw            

In [2]:
import pandas as pd
w = pd.read_parquet(r"C:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot\data\walkforward_2526.parquet")
p = pd.read_parquet(r"C:\Users\veers\OneDrive\Documents\FPL Agent\fpl-copilot\data\predictions_2526.parquet")

for gw in [1, 2, 20, 38]:
    a = w[w["gw"]==gw].set_index("element")["e_points"]
    b = p[p["gw"]==gw].set_index("element")["e_points"]
    j = a.to_frame("wf").join(b.to_frame("static"), how="inner")
    print(f"GW{gw}: n={len(j)}  identical={j['wf'].equals(j['static'])}  "
          f"corr={j['wf'].corr(j['static']):.4f}  mean_abs_diff={(j['wf']-j['static']).abs().mean():.4f}")

GW1: n=690  identical=False  corr=1.0000  mean_abs_diff=0.0006
GW2: n=705  identical=False  corr=0.9989  mean_abs_diff=0.0420
GW20: n=790  identical=False  corr=0.9979  mean_abs_diff=0.0501
GW38: n=841  identical=False  corr=0.9950  mean_abs_diff=0.0566


In [4]:
import sys; from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'squad'))
from simulator import load_season, simulate_season

season = load_season()
state, log = simulate_season(season, gws=[1, 2, 3])

GW 1   47 pts  (total   47)  transfer            -  bank 0.0  bench  3
GW 2   37 pts  (total   84)  transfer      413->17  bank 0.4  bench  0
GW 3   39 pts  (total  123)  transfer     235->119  bank 2.8  bench  7


In [5]:
log[["gw","points","raw_points","captain","captain_bonus","doubled_role","n_subs","bench_points"]]

,gw,points,raw_points,captain,captain_bonus,doubled_role,n_subs,bench_points
0,1,47,47,Mohamed Salah,8,captain,1,3
1,2,37,37,Mohamed Salah,5,captain,1,0
2,3,39,39,Mohamed Salah,3,captain,1,7


In [6]:
# 1. who blanked, and who came on
import pandas as pd
from simulator import gw_slice, gw_actuals
for gw in [1,2,3]:
    els = log.loc[log.gw==gw, "elements"].iloc[0]
    a = gw_actuals(season, gw).set_index("element")
    s = season[(season.gw==gw) & (season.element.isin(els))][["element","name","position","e_points","e_minutes","minutes","actual_points"]]
    print(f"--- GW{gw} zero-minute players in squad ---")
    print(s[s.minutes==0].to_string(index=False))

# 2. the actual squad in GW1
print(season[(season.gw==1) & (season.element.isin(log.elements.iloc[0]))]
      [["name","position","team","e_points","e_minutes","minutes","actual_points"]]
      .sort_values("e_points", ascending=False).to_string(index=False))

--- GW1 zero-minute players in squad ---
 element            name position  e_points  e_minutes  minutes  actual_points
     251 Nicolas Jackson      FWD  2.863668  48.783658        0              0
     403  Joško Gvardiol      DEF  4.411132  72.181022        0              0
--- GW2 zero-minute players in squad ---
 element            name position  e_points  e_minutes  minutes  actual_points
     235     Cole Palmer      MID  5.957800  86.741725        0              0
     251 Nicolas Jackson      FWD  0.723122   9.728192        0              0
     403  Joško Gvardiol      DEF  1.216439  17.413538        0              0
--- GW3 zero-minute players in squad ---
 element              name position  e_points  e_minutes  minutes  actual_points
     251   Nicolas Jackson      FWD  0.509917   4.487826        0              0
     403    Joško Gvardiol      DEF  0.747982   9.706832        0              0
     610 Aaron Wan-Bissaka      DEF  3.483016  83.650263        0              0


In [10]:
state, log = simulate_season(season)

GW 1   47 pts  (total   47)  transfer            -  bank 0.0  bench  3
GW 2   37 pts  (total   84)  transfer      413->17  bank 0.4  bench  0
GW 3   39 pts  (total  123)  transfer     235->119  bank 2.8  bench  7
GW 4   41 pts  (total  164)  transfer     403->257  bank 3.6  bench  1
GW 5   27 pts  (total  191)  transfer     402->317  bank 5.0  bench  0
GW 6   36 pts  (total  227)  transfer     610->403  bank 3.5  bench  4
GW 7   41 pts  (total  268)  transfer      250->64  bank 1.0  bench  0


ValueError: transfer unaffordable: bank 10 + 50 - 63 = -3

In [9]:
state, log = simulate_season(season, gws=[1,2,3])

GW 1   47 pts  (total   47)  transfer            -  bank 0.0  bench  3
GW 2   37 pts  (total   84)  transfer      413->17  bank 0.4  bench  0
GW 3   39 pts  (total  123)  transfer     235->119  bank 2.8  bench  7


In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'squad'))

from simulator import load_season, simulate_season
season = load_season()
state, log = simulate_season(season)

GW 1   47 pts  (total   47)  transfer            -  bank 0.0  bench  3
GW 2   37 pts  (total   84)  transfer      413->17  bank 0.4  bench  0


KeyboardInterrupt: 

In [1]:
from simulator import gw_slice, _adjusted_pool, decide_gameweek

ModuleNotFoundError: No module named 'simulator'

In [5]:
state30, _ = simulate_season(season, gws=list(range(1,31)), verbose=False)

gw = 31
pool = gw_slice(season, gw)
prices = dict(zip(pool["element"], pool["value"]))
adj = _adjusted_pool(pool, state30, prices)

print("pool size:", len(pool))
print("owned missing from pool:", [e for e in state30.elements if e not in set(pool.element)])
print("sell_value:", state30.sell_value(prices), "bank:", state30.bank,
      "power:", state30.budget(prices))
print("cost of keeping current 15 in adj prices:",
      int(adj.set_index('element').reindex(state30.elements)['value'].sum()))
print(adj.set_index('element').reindex(state30.elements)[['name','position','team','value']])

pool size: 664
owned missing from pool: [82, 1, 5, 267, 725]
sell_value: 964 bank: 1 power: 965
cost of keeping current 15 in adj prices: 643
                       name position         team  value
element                                                 
679          Mads Hermansen       GK     West Ham   42.0
593         Pape Matar Sarr      MID        Spurs   45.0
251         Nicolas Jackson      FWD      Chelsea   65.0
100           Junior Kroupi      FWD  Bournemouth   46.0
64            Ollie Watkins      FWD  Aston Villa   85.0
387      Dominik Szoboszlai      MID    Liverpool   68.0
82                      NaN      NaN          NaN    NaN
1                       NaN      NaN          NaN    NaN
374         Ibrahima Konaté      DEF    Liverpool   54.0
5                       NaN      NaN          NaN    NaN
267                     NaN      NaN          NaN    NaN
226         Trevoh Chalobah      DEF      Chelsea   55.0
381           Mohamed Salah      MID    Liverpool  140.0
343

In [6]:
missing = [82, 1, 5, 267, 725]
print(season[season.element.isin(missing)].groupby('element')['gw'].agg(['min','max','count']))
print("\npool sizes by gw:")
print(season.groupby('gw').size().to_string())

         min  max  count
element                 
1          1   38     37
5          1   38     37
82         1   38     36
267        1   38     37
725        4   38     34

pool sizes by gw:
gw
1     690
2     705
3     712
4     740
5     741
6     742
7     743
8     745
9     746
10    747
11    752
12    755
13    755
14    758
15    759
16    760
17    770
18    775
19    780
20    790
21    795
22    799
23    803
24    811
25    817
26    817
27    818
28    819
29    820
30    822
31    664
32    826
33    829
34    582
35    832
36    838
37    840
38    841


In [7]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'squad'))
from simulator import load_season, simulate_season
season = load_season()
state, log = simulate_season(season)

GW 1   47 pts  (total   47)  transfer            -  bank 0.0  bench  3
GW 2   37 pts  (total   84)  transfer      413->17  bank 0.4  bench  0
GW 3   39 pts  (total  123)  transfer     235->119  bank 2.8  bench  7
GW 4   41 pts  (total  164)  transfer     403->257  bank 3.6  bench  1
GW 5   27 pts  (total  191)  transfer     402->317  bank 5.0  bench  0
GW 6   36 pts  (total  227)  transfer     610->403  bank 3.5  bench  4
GW 7   41 pts  (total  268)  transfer      250->64  bank 1.0  bench  0
GW 8   52 pts  (total  320)  transfer     107->258  bank 1.0  bench  0
GW 9   59 pts  (total  379)  transfer     317->225  bank 0.0  bench  6
GW10   64 pts  (total  443)  transfer       17->20  bank 1.0  bench  4
GW11   53 pts  (total  496)  transfer       371->5  bank 0.1  bench  3
GW12   49 pts  (total  545)  transfer     225->436  bank 0.6  bench  2
GW13   39 pts  (total  584)  transfer       5->408  bank 1.5  bench  0
GW14   37 pts  (total  621)  transfer      20->387  bank 1.8  bench  2
GW15  

RuntimeError: GW31: no feasible squad found

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'squad'))
from simulator import load_season, simulate_season
season = load_season()
state, log = simulate_season(season)


GW 1   47 pts  (total   47)  transfer            -  bank 0.0  bench  3
GW 2   37 pts  (total   84)  transfer      413->17  bank 0.4  bench  0
GW 3   39 pts  (total  123)  transfer     235->119  bank 2.8  bench  7
GW 4   41 pts  (total  164)  transfer     403->257  bank 3.6  bench  1
GW 5   27 pts  (total  191)  transfer     402->317  bank 5.0  bench  0
GW 6   36 pts  (total  227)  transfer     610->403  bank 3.5  bench  4
GW 7   41 pts  (total  268)  transfer      250->64  bank 1.0  bench  0
GW 8   52 pts  (total  320)  transfer     107->258  bank 1.0  bench  0
GW 9   59 pts  (total  379)  transfer     317->225  bank 0.0  bench  6
GW10   64 pts  (total  443)  transfer       17->20  bank 1.0  bench  4
GW11   53 pts  (total  496)  transfer       371->5  bank 0.1  bench  3
GW12   49 pts  (total  545)  transfer     225->436  bank 0.6  bench  2
GW13   39 pts  (total  584)  transfer       5->408  bank 1.5  bench  0
GW14   37 pts  (total  621)  transfer      20->387  bank 1.8  bench  2
GW15  

In [2]:
log.to_parquet(r"data\simulation_log.parquet", index=False)

In [3]:
from baselines import run_all_baselines
table, rand = run_all_baselines(season)

set and forget      : 1368
hindsight ceiling   : 2554
random (50 squads)  : mean 816, sd 225, range 361-1272


In [4]:
from baselines import set_and_forget, _reassign_roles
from simulator import gw_slice
import pandas as pd

total, log = set_and_forget(season)
print("set-and-forget per-gw:", round(total/38, 1))
print(log[["gw","points","captain_bonus","doubled_role","n_subs","bench_points"]].head(10))
print("\ntotal bench points left behind:", log.bench_points.sum())
print("captain never doubled in", (log.doubled_role=='none').sum(), "gameweeks")

set-and-forget per-gw: 36.0
   gw  points  captain_bonus doubled_role  n_subs  bench_points
0   1      47              8      captain       1             3
1   2      37              5      captain       1             0
2   3      35              3      captain       1             7
3   4      45              9      captain       1             1
4   5      23              5      captain       0             0
5   6      27              2      captain       0             0
6   7      27              2      captain       1             0
7   8      44              2      captain       0             0
8   9      43              6      captain       1             0
9  10      47             10      captain       0             0

total bench points left behind: 82
captain never doubled in 0 gameweeks


In [5]:
from bootstrap import compare_strategies, report, season_total_interval, sensitivity_to_block_length
from baselines import set_and_forget

saf_total, saf_log = set_and_forget(season)
r = compare_strategies(log["points"].values, saf_log["points"].values, label="set_and_forget")
report(r)

vs set_and_forget
  observed margin : +0 points
  95% interval    : [+0, +0]
  model loses in  : 100.0% of resampled seasons
  block length 5, 2000 resamples
  -> LUCK CANNOT BE RULED OUT -- the interval includes zero


In [6]:
print("model log rows:", len(log), " total:", log["points"].sum())
print("saf log rows:  ", len(saf_log), " total:", saf_log["points"].sum())
print("identical:", (log["points"].values == saf_log["points"].values).all())

model log rows: 38  total: 1368
saf log rows:   38  total: 1368
identical: True


In [7]:
from simulator import simulate_season
state, model_log = simulate_season(season, verbose=False)
print("model:", model_log["points"].sum(), " saf:", saf_log["points"].sum())

model: 1889  saf: 1368


In [8]:
model_log.to_parquet(r"data\simulation_log.parquet", index=False)
saf_log.to_parquet(r"data\baseline_saf_log.parquet", index=False)

r = compare_strategies(model_log["points"].values, saf_log["points"].values, label="set_and_forget")
report(r)

vs set_and_forget
  observed margin : +521 points
  95% interval    : [+343, +708]
  model loses in  : 0.0% of resampled seasons
  block length 5, 2000 resamples
  -> the margin is unlikely to be luck


In [9]:
print(sensitivity_to_block_length(model_log["points"].values, saf_log["points"].values).to_string(index=False))

 block_length  margin  ci_low  ci_high  excludes_zero  loses_pct
            1   521.0   368.0    664.0           True        0.0
            2   521.0   347.0    689.0           True        0.0
            4   521.0   347.0    693.0           True        0.0
            5   521.0   343.0    708.0           True        0.0
            6   521.0   337.0    705.0           True        0.0
            8   521.0   340.0    702.0           True        0.0
           10   521.0   347.0    694.0           True        0.0


In [10]:
from baselines import hindsight_set_and_forget
hind_total, hind_log = hindsight_set_and_forget(season)
hind_log.to_parquet(r"data\baseline_hindsight_log.parquet", index=False)

r2 = compare_strategies(model_log["points"].values, hind_log["points"].values, label="hindsight_ceiling")
report(r2)

vs hindsight_ceiling
  observed margin : -665 points
  95% interval    : [-958, -386]
  model loses in  : 100.0% of resampled seasons
  block length 5, 2000 resamples
  -> the margin is unlikely to be luck


In [11]:
import pulp
print([s for s in pulp.listSolvers(onlyAvailable=True)])

['PULP_CBC_CMD', 'HiGHS']


In [12]:
import time, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'squad'))
from optimize import load_gw_data, optimize_squad
import pulp

df = load_gw_data(gw=1)

t = time.time()
prob_cbc, _ = optimize_squad(df, mode="balanced")
cbc = time.time() - t

# same problem, HiGHS
t = time.time()
prob_h = pulp.LpProblem("x", pulp.LpMaximize)
# rebuild via optimize_squad but solve with HiGHS -- quick hack for timing
import optimize
orig = pulp.PULP_CBC_CMD
pulp.PULP_CBC_CMD = lambda **kw: pulp.HiGHS(msg=False)
prob_hi, _ = optimize_squad(df, mode="balanced")
pulp.PULP_CBC_CMD = orig
hi = time.time() - t

print(f"CBC   {cbc:.2f}s  obj {pulp.value(prob_cbc.objective):.4f}")
print(f"HiGHS {hi:.2f}s  obj {pulp.value(prob_hi.objective):.4f}")

CBC   1.24s  obj 61.2512
HiGHS 0.64s  obj 61.2512


In [1]:
import importlib, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'squad'))
from simulator import load_season, gw_slice
from transfer_mip import build_and_solve
import time

season = load_season()
pools = {gw: gw_slice(season, gw) for gw in [1,2,3,4,5,6]}

t = time.time()
status, plan = build_and_solve(pools, current_squad=[], purchase_prices={}, bank=0, free_transfers=1)
print(status, f"{time.time()-t:.1f}s")
for p in plan:
    print(p["gw"], "buys", len(p["buys"]), "sells", len(p["sells"]), "hits", p["hits"], f"decay {p['decay_weight']:.2f}")

Optimal 32.6s
1 buys 15 sells 0 hits 0 decay 1.00
2 buys 1 sells 1 hits 0 decay 0.85
3 buys 1 sells 1 hits 0 decay 0.72
4 buys 1 sells 1 hits 0 decay 0.61
5 buys 1 sells 1 hits 0 decay 0.52
6 buys 1 sells 1 hits 0 decay 0.44


In [2]:
# 1. Does it ever choose to take a hit if the payoff is there?
status2, plan2 = build_and_solve(pools, current_squad=[], purchase_prices={},
                                 bank=0, free_transfers=1, decay=1.0)
print("no decay:", [(p["gw"], len(p["buys"]), p["hits"]) for p in plan2])

# 2. How does solve time scale with horizon?
for H in [2, 4, 6]:
    pp = {gw: gw_slice(season, gw) for gw in range(1, 1+H)}
    t = time.time()
    s, _ = build_and_solve(pp, [], {}, 0, 1)
    print(f"H={H}: {time.time()-t:.1f}s")

no decay: [(1, 15, 0), (2, 1, 0), (3, 1, 0), (4, 1, 0), (5, 1, 0), (6, 1, 0)]
H=2: 6.5s
H=4: 11.7s
H=6: 33.1s


In [3]:
# give it 5 free transfers up front - if the projection is the constraint,
# it should now make several transfers in GW2
s, p = build_and_solve(pools, current_squad=[], purchase_prices={},
                       bank=0, free_transfers=5, decay=1.0)
print([(x["gw"], len(x["buys"]), x["hits"], x["free_transfers_assumed"]) for x in p])

[(1, 15, 0, 5), (2, 5, 0, 5), (3, 5, 0, 5), (4, 5, 0, 5), (5, 5, 0, 5), (6, 5, 0, 5)]


In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'squad'))

from simulator import load_season, gw_slice
from transfer_mip import build_and_solve

season = load_season()
pools = {gw: gw_slice(season, gw) for gw in [1, 2, 3, 4, 5, 6]}

s, p = build_and_solve(pools, [], {}, 0, free_transfers=5, decay=1.0)
print("ft=5:", s, [(x["gw"], x["transfers_made"], x["hits"], x["free_transfers"]) for x in p])

s2, p2 = build_and_solve(pools, [], {}, 0, free_transfers=1, decay=0.85)
print("ft=1:", s2, [(x["gw"], x["transfers_made"], x["hits"], x["free_transfers"]) for x in p2])

ft=5: Optimal [(1, 15, 0, 5), (2, 1, 0, 5), (3, 2, 0, 5), (4, 3, 0, 4), (5, 1, 0, 2), (6, 2, 0, 2)]
ft=1: Optimal [(1, 15, 0, 1), (2, 2, 0, 2), (3, 1, 0, 1), (4, 1, 0, 1), (5, 1, 0, 1), (6, 1, 0, 1)]


In [2]:
s3, p3 = build_and_solve(pools, [], {}, 0, free_transfers=0, decay=1.0)
print("ft=0:", [(x["gw"], x["transfers_made"], x["hits"], x["free_transfers"]) for x in p3])

ft=0: [(1, 15, 0, 0), (2, 1, 0, 1), (3, 1, 0, 1), (4, 1, 0, 1), (5, 1, 0, 1), (6, 1, 0, 1)]


In [3]:
# GW2 alone: start with a squad, zero free transfers, so any transfer costs 4
_, p1 = build_and_solve({1: pools[1]}, [], {}, 0, free_transfers=1)
squad1 = p1[0]["squad"]
pp = {e: int(pools[1].set_index("element").loc[e, "value"]) for e in squad1}

for ft in [0, 1]:
    s, p = build_and_solve({2: pools[2]}, squad1, pp, bank=0, free_transfers=ft)
    print(f"ft={ft}:", s, "transfers", p[0]["transfers_made"], "hits", p[0]["hits"])

ft=0: Optimal transfers 0 hits 0
ft=1: Optimal transfers 1 hits 0


In [4]:
import pandas as pd
# make one unowned player absurdly good - worth far more than a -4
pool2 = pools[2].copy()
target = pool2[~pool2.element.isin(squad1)].iloc[0]
pool2.loc[pool2.element == target.element, "e_points"] = 50.0

s, p = build_and_solve({2: pool2}, squad1, pp, bank=0, free_transfers=0)
print(s, "transfers", p[0]["transfers_made"], "hits", p[0]["hits"])
print("bought the 50-pointer:", target.element in p[0]["buys"])

Optimal transfers 2 hits 2
bought the 50-pointer: True


In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'squad'))
from simulator import load_season, simulate_season

season = load_season()
gws = list(range(1, 9))

s1, l1 = simulate_season(season, gws=gws, policy="single", verbose=False)
print("single:", s1.total_points)

t = time.time()
s2, l2 = simulate_season(season, gws=gws, policy="mip", horizon=6, verbose=False)
print(f"mip H=6: {s2.total_points}  ({time.time()-t:.0f}s)")
print(l2[["gw","points","n_transfers","hit","effective_horizon"]].to_string(index=False))

single: 320
mip H=6: 452  (39s)
 gw  points  n_transfers  hit  effective_horizon
  1      47            0    0                  1
  2      37            5   12                  6
  3      52            1    0                  6
  4      83            1    0                  5
  5      42            1    0                  4
  6      73            1    0                  3
  7      54            1    0                  2
  8      64            1    0                  1


In [2]:
import pandas as pd
from scipy.stats import spearmanr

wf = pd.read_parquet(r"data\walkforward_h6_2526.parquet")
v = wf.dropna(subset=["actual_points","e_points"]).copy()
v["actual_points"] = pd.to_numeric(v["actual_points"], errors="coerce")

bands = {
    "All rows":        lambda d: d,
    "Played (>0)":     lambda d: d[d.minutes > 0],
    "Started (60+)":   lambda d: d[d.minutes >= 60],
}

print(f"{'step':<6}" + "".join(f"{b:>18}" for b in bands))
for step in sorted(v.horizon_step.unique()):
    s = v[v.horizon_step == step]
    row = f"{step:<6}"
    for f in bands.values():
        d = f(s)
        row += f"{spearmanr(d.e_points, d.actual_points).correlation:>10.3f} (n={len(d)//1000}k)"
    print(row)

step            All rows       Played (>0)     Started (60+)
0          0.715 (n=29k)     0.337 (n=11k)     0.099 (n=7k)
1          0.682 (n=28k)     0.294 (n=10k)     0.087 (n=7k)
2          0.661 (n=27k)     0.276 (n=10k)     0.097 (n=7k)
3          0.641 (n=26k)     0.249 (n=10k)     0.080 (n=6k)
4          0.629 (n=25k)     0.236 (n=9k)     0.076 (n=6k)
5          0.617 (n=24k)     0.221 (n=9k)     0.068 (n=6k)


In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'squad'))
from simulator import load_season, simulate_season

season_h = load_season(horizon_aware=True)
gws = list(range(1, 9))

s1, l1 = simulate_season(season_h, gws=gws, policy="single", verbose=False)
print("single:", s1.total_points)

t = time.time()
s2, l2 = simulate_season(season_h, gws=gws, policy="mip", horizon=6, verbose=False)
print(f"mip H=6: {s2.total_points}  ({time.time()-t:.0f}s)")

single: 320


PulpError: Cannot multiply variables with NaN/inf values

In [2]:
import pandas as pd
season_h = load_season(horizon_aware=True)

print("NaN e_points:", season_h["e_points"].isna().sum(), "of", len(season_h))
print("\nby horizon_step:")
print(season_h.groupby("horizon_step")["e_points"].apply(lambda s: s.isna().sum()))
print("\nsample rows:")
print(season_h[season_h["e_points"].isna()].head(5)[
    ["element","name","cutoff","gw","horizon_step","e_minutes","p_start","npxg90"]])

NaN e_points: 3701 of 165401

by horizon_step:
horizon_step
0      0
1    560
2    694
3    782
4    912
5    753
Name: e_points, dtype: int64

sample rows:
      element                   name  cutoff  gw  horizon_step  e_minutes  \
4140      691  Dominic Calvert-Lewin       1   2             1        NaN   
4141      691  Dominic Calvert-Lewin       1   3             2        NaN   
4142      691  Dominic Calvert-Lewin       1   4             3        NaN   
4143      691  Dominic Calvert-Lewin       1   5             4        NaN   
4144      691  Dominic Calvert-Lewin       1   6             5        NaN   

      p_start    npxg90  
4140      NaN  0.465205  
4141      NaN  0.465205  
4142      NaN  0.465205  
4143      NaN  0.465205  
4144      NaN  0.465205  
